In [1]:
"""
The purpose of this Jupyter notebook is to perform control-based
Z-scoring of the Dharmacon pooled screen subset.

The rationale behind using control-based Z-scores rather than Z-scores
taking all wells into account is the following: Z-scores depend on the
mean and the standard deviation of the plates, which may differ greatly
depending on the precise plate content. For instance, if a plate has
many genes the knockdown of which facilitates the VACV life cycle, this
plate's mean will be significantly greater than that of a plate having a
more balanced composition. Thus, a gene facilitating VACV infection will
have a moderate Z-score even though it would have an extraordinarily
high Z-score on the balanced plate.

Control-based Z-scoring anchors all plates to the same baseline and
removes composition bias, thus making cross-plate comparison of Z-scores
legitimate.

Control-based Z-scoring is possible since all plates contain the same
controls and all controls occur the same number of times on each plate.
"""

"\nThe purpose of this Jupyter notebook is to perform control-based\nZ-scoring of the Dharmacon pooled screen subset.\n\nThe rationale behind using control-based Z-scores rather than Z-scores\ntaking all wells into account is the following: Z-scores depend on the\nmean and the standard deviation of the plates, which may differ greatly\ndepending on the precise plate content. For instance, if a plate has\nmany genes the knockdown of which facilitates the VACV life cycle, this\nplate's mean will be significantly greater than that of a plate having a\nmore balanced composition. Thus, a gene facilitating VACV infection will\nhave a moderate Z-score even though it would have an extraordinarily\nhigh Z-score on the balanced plate.\n\nControl-based Z-scoring anchors all plates to the same baseline and\nremoves composition bias, thus making cross-plate comparison of Z-scores\nlegitimate.\n\nControl-based Z-scoring is possible since all plates contain the same\ncontrols and all controls occur t

In [2]:
import numpy as np
import pandas as pd

### Loading the Data Set

In [3]:
screen_subset_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Processing_Dharmacon_"
    "pooled_genome_1_and_2_subset/Dharmacon_pooled_G1_G2_screening_"
    "plates_subset_with_missing_UniProt_IDs.tsv"
)

screen_subset_df = pd.read_csv(
    screen_subset_path,
    sep="\t"
)

### Performing Control-Based Z-Scoring

In [7]:
# The means and standard deviations of the individual plates differ
# quite a lot, as determined in the notebook
# "plate_investigation.ipynb", which is why control-based Z-scoring is
# performed
# This is possible since when excluding the "UNKNOWN" control, all
# plates contain the same controls and all controls also occur the same
# number of times on each plate
# In control-based Z-scoring, the mean and standard deviation are
# computed based on control wells only

# Both the intensity values and the cell counts are Z-scored
# Define the raw features to be Z-scored
raw_features_list = [
    "eCount_oCells",
    "dIntensity_cPathogen_eMean_oNuclei",
    "dIntensity_cPathogen_eMean_oPeriNuclei",
    "dIntensity_cPathogen_eMean_oCells",
    "dIntensity_cPathogen_eMean_oVoronoiCells",
    "dIntensity_cLatePathogen_eMean_oNuclei",
    "dIntensity_cLatePathogen_eMean_oPeriNuclei",
    "dIntensity_cLatePathogen_eMean_oCells",
    "dIntensity_cLatePathogen_eMean_oVoronoiCells"
]

In [8]:
# Identify unique plate IDs, which are stored in the `Barcode` column
plate_ids = screen_subset_df["Barcode"].unique()

In [9]:
# Create a copy of the screen subset DataFrame so as not to overwrite
# the original one
control_based_Z_scored_screen_subset_df = screen_subset_df.copy()

In [10]:
# Control-based Z-scoring is done on a per plate basis and also for each
# feature individually

# Iterate over the individual plates
for plate_id in plate_ids:
    # Iterate over the individual features to be Z-scored
    for raw_feature in raw_features_list:
        # Determine the mean as well as the standard deviation of the
        # current feature in the current plate based on controll wells
        # only
        # Do not forget to exclude the "UNKNOWN" control!
        current_mean = screen_subset_df.loc[
            (screen_subset_df["Barcode"] == plate_id)
            &
            (screen_subset_df["WellType"] == "CONTROL")
            &
            (screen_subset_df["ID_openBIS"] != "UNKNOWN"),
            raw_feature
        ].mean()
        current_std = screen_subset_df.loc[
            (screen_subset_df["Barcode"] == plate_id)
            &
            (screen_subset_df["WellType"] == "CONTROL")
            &
            (screen_subset_df["ID_openBIS"] != "UNKNOWN"),
            raw_feature
        ].std()

        # Perform the control-based Z-scoring
        Z_scored_feature = raw_feature + "_nZScore"

        control_based_Z_scored_screen_subset_df.loc[
            control_based_Z_scored_screen_subset_df["Barcode"] == plate_id,
            Z_scored_feature
        ] = (
            screen_subset_df.loc[
                screen_subset_df["Barcode"] == plate_id,
                raw_feature
            ]
            -
            current_mean
        ) / current_std

In [12]:
# Save the control-based Z-scored DataFrame to a TSV file
control_based_Z_scored_screen_subset_df.to_csv(
    "Dharmacon_pooled_G1_G2_screening_plates_subset_with_missing_"
    "UniProt_IDs_control-based_Z-scored.tsv",
    sep="\t",
    index=False
)

### Evaluating Effect of Control-Based Z-Scoring

In [9]:
# Investigate whether control-based Z-scoring reduces plate-to-plate
# variation of controls
# First, compute the plate-to-plate variation before Z-scoring
# This is done in the following manner: For each control, the mean of
# the intensity is computed per plate
# In a subsequent step, the standard deviation of the per plate means is
# computed for each control

# Determine the unique controls
unique_controls = screen_subset_df.loc[
    screen_subset_df["WellType"] == "CONTROL",
    "ID_openBIS"
].unique().tolist()

# Remove the "UNKNOWN" control
unique_controls.remove("UNKNOWN")

assert "UNKNOWN" not in unique_controls, (
    "The 'UNKNOWN' control is still present!"
)

std_per_control_before_Z_scoring = []

# Iterate over the controls
for control in unique_controls:
    # Extract the current control, group the resulting DataFrame by
    # plates and compute the mean intensity per plate
    mean_int_per_plate = (
        screen_subset_df.loc[
            screen_subset_df["ID_openBIS"] == control,
            ["Barcode", "dIntensity_cPathogen_eMean_oVoronoiCells"]
        ]
        .set_index("Barcode")
        .groupby(level="Barcode")
        .mean()
    )

    # Finally, compute the standard deviation across all plates for that
    # control
    control_std = mean_int_per_plate.std().iloc[0]
    std_per_control_before_Z_scoring.append(control_std)

In [3]:
# Now, repeat the procedure for the control intensities after
# control-based Z-scoring
# Load the corresponding TSV file into a DataFrame
control_based_Z_scored_screen_subset_path = (
    "Dharmacon_pooled_G1_G2_screening_plates_subset_with_missing_"
    "UniProt_IDs_control-based_Z-scored.tsv"
)

control_based_Z_scored_screen_subset_df = pd.read_csv(
    control_based_Z_scored_screen_subset_path,
    sep="\t"
)

In [11]:
std_per_control_after_Z_scoring = []

# Iterate over the controls
for control in unique_controls:
    # Extract the current control, group the resulting DataFrame by
    # plates and compute the mean intensity per plate
    mean_int_per_plate = (
        control_based_Z_scored_screen_subset_df.loc[
            control_based_Z_scored_screen_subset_df["ID_openBIS"]
            ==
            control,
            ["Barcode", "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore"]
        ]
        .set_index("Barcode")
        .groupby(level="Barcode")
        .mean()
    )

    # Finally, compute the standard deviation across all plates for that
    # control
    control_std = mean_int_per_plate.std().iloc[0]
    std_per_control_after_Z_scoring.append(control_std)

In [12]:
assert all(
    np.array(std_per_control_before_Z_scoring)
    >
    np.array(std_per_control_after_Z_scoring)
), (
    "Control-based Z-scoring does not reduce the standard deviation in "
    "absolute terms!"
)

AssertionError: Control-based Z-scoring does not reduce the standard deviation in absolute terms!

In [13]:
# As it turns out, control-based Z-scoring does not reduce the standard
# deviation in absolute terms
# However, there is the possibility that the standard deviation is
# reduced in relative terms
# To evaluate whether this is the case, the following is done
# For each control, the ratio is computed between the plate-to-plate
# standard deviation and the average standard deviation within plates

# The plate-to-plate standard deviations have already been computed
# Thus, all that remains to do is compute the average standard deviation
# within plates

# Do this for the intensities before control-based Z-scoring
std_within_plate_before_Z_scoring = []

for control in unique_controls:
    std_per_plate = (
        screen_subset_df.loc[
            screen_subset_df["ID_openBIS"] == control,
            ["Barcode", "dIntensity_cPathogen_eMean_oVoronoiCells"]
        ]
        .set_index("Barcode")
        .groupby(level="Barcode")
        .std()
    )

    average_std = std_per_plate.mean().iloc[0]
    std_within_plate_before_Z_scoring.append(average_std)

In [14]:
# Repeat the procedure for the control intensities after control-based
# Z-scoring
std_within_plate_after_Z_scoring = []

for control in unique_controls:
    std_per_plate = (
        control_based_Z_scored_screen_subset_df.loc[
            control_based_Z_scored_screen_subset_df["ID_openBIS"]
            ==
            control,
            ["Barcode", "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore"]
        ]
        .set_index("Barcode")
        .groupby(level="Barcode")
        .std()
    )

    average_std = std_per_plate.mean().iloc[0]
    std_within_plate_after_Z_scoring.append(average_std)

In [15]:
# Now, compute the ratios between the plate-to-plate standard deviation
# and the average standard deviation within plates
# To this end, vectorized operations offered by NumPy are leveraged
ratio_per_control_before_Z_scoring = (
    np.array(std_per_control_before_Z_scoring)
    /
    np.array(std_within_plate_before_Z_scoring)
)

ratio_per_control_after_Z_scoring = (
    np.array(std_per_control_after_Z_scoring)
    /
    np.array(std_within_plate_after_Z_scoring)
)

In [16]:
assert all(
    ratio_per_control_before_Z_scoring
    >
    ratio_per_control_after_Z_scoring
), (
    "Control-based Z-scoring does not reduce the standard deviation in "
    "relative terms either!"
)

### Filling Missing "Name" Cells

In [6]:
# Some cells in the "Name" column are empty; they are filled in
# However, prior to that, a couple of investigations are performed

# Check whether all empty cells belong to controls
empty_well_type = control_based_Z_scored_screen_subset_df.loc[
    control_based_Z_scored_screen_subset_df["Name"].isna(),
    "WellType"
].unique()

assert len(empty_well_type) == 1, (
    "Empty 'Name' cells belong to more than one well type!"
)

AssertionError: Empty 'Name' cells belong to more than one well type!

In [7]:
print(empty_well_type)

['CONTROL' 'POOLED_SIRNA']


In [8]:
# As it turns out, not all empty cells belong to controls
# Only empty cells belonging to controls are filled
control_based_Z_scored_screen_subset_df.loc[
    control_based_Z_scored_screen_subset_df["WellType"] == "CONTROL",
    "Name"
] = control_based_Z_scored_screen_subset_df.loc[
    control_based_Z_scored_screen_subset_df["WellType"] == "CONTROL",
    "Name"
].fillna(
    control_based_Z_scored_screen_subset_df.loc[
        control_based_Z_scored_screen_subset_df["WellType"] == "CONTROL",
        "ID_openBIS"
    ]
)

In [13]:
# As a sanity check, verify that names are provided for all controls
assert control_based_Z_scored_screen_subset_df.loc[
    control_based_Z_scored_screen_subset_df["WellType"] == "CONTROL",
    "Name"
].notna().all(), (
    "For some controls, some 'Name' entries are still empty!"
)

In [14]:
# Overwrite the TSV file with the updated version
control_based_Z_scored_screen_subset_df.to_csv(
    "Dharmacon_pooled_G1_G2_screening_plates_subset_with_missing_"
    "UniProt_IDs_control-based_Z-scored.tsv",
    sep="\t",
    index=False
)